# Locator map generator

Makes a three-panel map for a Cognizant Foundation proposal: India with the state
highlighted, the state with the district highlighted, and the district with the
block and your site points.

**How to use this notebook**

1. Run the setup cell once. It takes a few minutes the first time.
2. Fill in the form in the second cell. You do not need to edit any code.
3. Run the last cell to draw the map and download the files.

Every boundary comes from a published government dataset. Nothing is drawn or
guessed. If something cannot be confirmed the notebook tells you and stops.


## Step 1. Setup

Run this once per session.

In [ ]:
#@title Set up the tool { display-mode: "form" }
#@markdown Leave this as it is unless you have been given a different address.
repo_url = "https://github.com/YOUR-ORG/locator-maps.git"  #@param {type:"string"}

import os, subprocess, sys

FOLDER = "locator-maps"

if not os.path.isdir(FOLDER):
    print("Getting the tool...")
    subprocess.run(["git", "clone", "--depth", "1", repo_url, FOLDER], check=True)

os.chdir("/content/" + FOLDER if os.path.isdir("/content/" + FOLDER) else FOLDER)

print("Installing what it needs...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

print("Downloading boundary data (about 150 MB, once per session)...")
from locator import data as _data
for _key in _data.DATASETS:
    _data.ensure_dataset(_key, log=print)

print("
Ready.")


## Step 2. Describe the map you want

Fill in the boxes. For the site coordinates, open Google Maps, right-click the
exact spot, and click the pair of numbers at the top of the menu to copy them.
Paste the first number into **lat** and the second into **lon**.

Leave a site's name empty if you do not need it.

In [ ]:
#@title Map details { display-mode: "form" }

state = "Tamil Nadu"  #@param {type:"string"}
district = "Madurai"  #@param {type:"string"}
#@markdown Blocks to highlight. Separate several with commas.
blocks = "Madurai West"  #@param {type:"string"}

#@markdown ---
#@markdown ### Site 1
site1_name = "Government Rajaji Hospital (GRH)"  #@param {type:"string"}
site1_lat = 9.9195  #@param {type:"number"}
site1_lon = 78.1193  #@param {type:"number"}
site1_type = "hospital"  #@param ["hospital", "school", "camp"]

#@markdown ### Site 2
site2_name = ""  #@param {type:"string"}
site2_lat = 0  #@param {type:"number"}
site2_lon = 0  #@param {type:"number"}
site2_type = "school"  #@param ["hospital", "school", "camp"]

#@markdown ### Site 3
site3_name = ""  #@param {type:"string"}
site3_lat = 0  #@param {type:"number"}
site3_lon = 0  #@param {type:"number"}
site3_type = "camp"  #@param ["hospital", "school", "camp"]

#@markdown ---
#@markdown Leave the title empty to build it from the state and district.
title = ""  #@param {type:"string"}
output_name = "locator_map"  #@param {type:"string"}
#@markdown Tick this only after reading a warning and deciding it is fine.
ignore_warnings = False  #@param {type:"boolean"}

config = {
    "state": state.strip(),
    "district": district.strip(),
    "blocks": [b.strip() for b in blocks.split(",") if b.strip()],
    "sites": [
        {"name": n, "lat": la, "lon": lo, "type": t}
        for n, la, lo, t in [
            (site1_name, site1_lat, site1_lon, site1_type),
            (site2_name, site2_lat, site2_lon, site2_type),
            (site3_name, site3_lat, site3_lon, site3_type),
        ]
        if n.strip()
    ],
    "title": title.strip() or None,
    "output_name": output_name.strip() or "locator_map",
}

print("This is what will be drawn:")
for line in [
    f"  State:    {config['state']}",
    f"  District: {config['district']}",
    f"  Blocks:   {', '.join(config['blocks']) or '(none highlighted)'}",
]:
    print(line)
for s in config["sites"]:
    print(f"  Site:     {s['name']} at ({s['lat']}, {s['lon']})")


## Step 3. Draw it

This checks everything first. If a name or a coordinate cannot be confirmed it
stops and tells you what to fix, rather than drawing something wrong.

In [ ]:
#@title Make the map { display-mode: "form" }
from pathlib import Path
from IPython.display import Image, display as show

from locator.cli import run_from_config

outcome = run_from_config(config, force=ignore_warnings)
report = outcome["report"]

print(report.render_text())
print()
print("Result:", report.summary())
print()

if not outcome["rendered"]:
    if report.errors:
        print("Nothing was drawn. Fix the points marked ERROR above and run again.")
    else:
        print(
            "Nothing was drawn because of the warnings above.
"
            "If you have read them and they are fine, tick 'ignore_warnings'
"
            "in the form above and run this cell again."
        )
else:
    preview = Path(outcome["output_dir"]) / f"{config['output_name']}_300dpi.png"
    show(Image(filename=str(preview)))
    print("Files made:")
    for f in outcome["files"]:
        print("  ", f.name)


## Step 4. Save the files

Run **one** of the two cells below.

In [ ]:
#@title Download the files to your computer { display-mode: "form" }
from google.colab import files

if not outcome.get("rendered"):
    print("There is nothing to download yet.")
else:
    for f in outcome["files"]:
        files.download(str(f))


In [ ]:
#@title Or save them to Google Drive { display-mode: "form" }
#@markdown Folder inside your Drive. It is created if it does not exist.
drive_folder = "Locator maps"  #@param {type:"string"}

import shutil
from pathlib import Path
from google.colab import drive

if not outcome.get("rendered"):
    print("There is nothing to save yet.")
else:
    drive.mount("/content/drive", force_remount=False)
    destination = Path("/content/drive/MyDrive") / drive_folder / config["output_name"]
    destination.mkdir(parents=True, exist_ok=True)
    for f in outcome["files"]:
        shutil.copy2(f, destination / f.name)
        print("Saved", f.name)
    print("
All files are in Drive under:", destination)
